# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 23_reused_stock_dataset_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-24
### Properties script
**Goal:** generating a dataset for the reused stock that serves as part of the geometric input for the cost matrix
**Inputs:**
*   parameters for properties of the stock
*   search space of geometry

**Outputs:**
*   CSV file with dataset of reused stock

## Mounting Google drive

In [ ]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

## Start script

In [ ]:
import pandas as pd
import numpy as np
import random

# 1. DEFINIEER HET DONOR GEBOUW (PAKHUIS SCENARIO)
donor_batches = [
    {"batch_id": "B01", "count": 24, "orig_width": 180, "orig_thickness": 600, "orig_length": 12000},
    {"batch_id": "B02", "count": 150, "orig_width": 75, "orig_thickness": 225, "orig_length": 5400},
    {"batch_id": "B03", "count": 30, "orig_width": 200, "orig_thickness": 200, "orig_length": 4200}
]

# 2. MECHANISCHE EIGENSCHAPPEN (NEN-EN 338 normen naaldhout)
mechanical_props = {
    'C24': {'e_mod': 11000, 'f_mk': 24, 'density': 420},
    'C18': {'e_mod': 9000,  'f_mk': 18, 'density': 380}
}

# 3. DEFINIEER DE VERWERKINGS LOGICA
def process_element(element_data, global_index):
    # Identificatie: Format RS_0000
    member_id = f"RS_{global_index:04d}"

    # Geometrie: Sloop- en schaafverlies
    cut_loss = random.randint(100, 400)
    length_actual = element_data['orig_length'] - cut_loss
    width = element_data['orig_width'] - random.randint(10, 16)
    thickness = element_data['orig_thickness'] - random.randint(10, 16)

    # Grading (Kwaliteit) bepalen
    grade = np.random.choice(['C24', 'C18'], p=[0.60, 0.40])

    # Koppel mechanische eigenschappen
    e_modulus_eff = mechanical_props[grade]['e_mod']
    f_mk = mechanical_props[grade]['f_mk']
    density = mechanical_props[grade]['density']

    # LCA Aannames
    transport_dist = random.randint(20, 150)

    return {
        # Identificatie geüpdatet conform jouw matrix
        "Member_ID": member_id,
        "State": 1,          # 1 = Reclaimed (0 zou bijv. Virgin EWP kunnen zijn)

        # Geometrie
        "Length_Actual": length_actual,
        "Width": width,
        "Thickness": thickness,

        # Mechanisch
        "E_modulus_eff": e_modulus_eff,
        "f_mk": f_mk,
        "Density": density,

        # LCA
        "Embodied Carbon Coëfficiënt": 15.0,
        "Transport_Dist": transport_dist,
        "Emmisiefactor": 0.062,
        "Bewerkingsfactor": 1
    }

# 4. GENEREREN VAN DE DATASET MET GLOBALE TELLER
inventory_list = []
current_id_number = 1

for batch in donor_batches:
    for _ in range(batch['count']):
        processed_element = process_element(batch, current_id_number)
        inventory_list.append(processed_element)
        current_id_number += 1  # Verhoog de teller voor het volgende element

# Maak het DataFrame
df_cost_matrix = pd.DataFrame(inventory_list)

# Resultaat bekijken
print(df_cost_matrix[['Member_ID', 'State', 'Length_Actual', 'Width', 'Thickness', 'f_mk']].head())